In [1]:
!nvidia-smi
import torch
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CUDA not available")


/bin/bash: line 1: nvidia-smi: command not found
CUDA not available


In [ ]:
REPO_URL = "https://github.com/dayan-battulga/FER_NLP.git"  # TODO: replace with your repo URL
REPO_DIR = "FER_NLP"  # TODO: adjust if your cloned folder name differs
!git clone "$REPO_URL" "$REPO_DIR"
%cd /content/$REPO_DIR


Cloning into 'dunedain_ner'...
fatal: unable to access 'https://github.com/<ORG>/<REPO>.git/': The requested URL returned error: 400
[Errno 2] No such file or directory: '/content/dunedain_ner'
/Users/dayanbattulga/Desktop/personal-code/misc/dunedain_ner/notebooks


In [3]:
!pip install -r requirements.txt


ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")
DRIVE_RESULTS_DIR = "/content/drive/MyDrive"  # TODO: adjust path if needed
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

if os.path.lexists("results"):
    !rm -rf results
!ln -s "$DRIVE_RESULTS_DIR" results
!ls -la results


ModuleNotFoundError: No module named 'google.colab'

In [ ]:
import os
import subprocess

# TODO: set WANDB_API_KEY in Colab Secrets and expose it to env vars.
WANDB_API_KEY = os.environ.get("WANDB_API_KEY")
if not WANDB_API_KEY:
    raise RuntimeError("WANDB_API_KEY is not set. Configure Colab Secrets before continuing.")
subprocess.run(["wandb", "login", WANDB_API_KEY], check=True)


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/deberta-v3-large",
    use_fast=True,
    add_prefix_space=True,
)
enc = tokenizer(
    [["Apple", "Inc."]],
    is_split_into_words=True,
    truncation=True,
    max_length=32,
)
print("Apple Inc. subwords:", tokenizer.convert_ids_to_tokens(enc["input_ids"][0]))
print("Apple Inc. word_ids:", enc.word_ids(batch_index=0))
!python -m src.data


In [ ]:
# 1-epoch smoke run (seed 88) -> results/deberta_smoke_seed88
!python -m src.train --config configs/baseline/deberta_smoke.yaml


In [ ]:
import yaml
import subprocess

tmp_cfg = "/tmp/deberta_efficient.yaml"
cfg = yaml.safe_load(open("configs/baseline/deberta_efficient.yaml", "r", encoding="utf-8"))
cfg["seeds"] = [88]
with open(tmp_cfg, "w", encoding="utf-8") as handle:
    yaml.safe_dump(cfg, handle, sort_keys=False)
subprocess.run(["python", "-m", "src.train", "--config", tmp_cfg], check=True)


In [ ]:
import yaml
import subprocess

tmp_cfg = "/tmp/deberta_efficient.yaml"
cfg = yaml.safe_load(open("configs/baseline/deberta_efficient.yaml", "r", encoding="utf-8"))
cfg["seeds"] = [5768]
with open(tmp_cfg, "w", encoding="utf-8") as handle:
    yaml.safe_dump(cfg, handle, sort_keys=False)
subprocess.run(["python", "-m", "src.train", "--config", tmp_cfg], check=True)


In [ ]:
import yaml
import subprocess

tmp_cfg = "/tmp/deberta_efficient.yaml"
cfg = yaml.safe_load(open("configs/baseline/deberta_efficient.yaml", "r", encoding="utf-8"))
cfg["seeds"] = [78516]
with open(tmp_cfg, "w", encoding="utf-8") as handle:
    yaml.safe_dump(cfg, handle, sort_keys=False)
subprocess.run(["python", "-m", "src.train", "--config", tmp_cfg], check=True)


In [ ]:
from pathlib import Path

aggregate_path = Path("results/deberta_efficient_aggregate.json")
if aggregate_path.exists():
    print(aggregate_path.read_text(encoding="utf-8"))
else:
    print(f"{aggregate_path} not found yet.")
